# Assignment 1 — Custom Missing-Value Imputer

**Course:** Feature Engineering & MLOps  
**Topic:** Custom Imputer Class (Missing Value Handling)

## Objective
Design and implement a custom Python imputer following scikit-learn's `fit()` / `transform()` pattern, apply it to the supplied `student_performance_raw.csv` dataset, verify its behavior and compare its numeric fill values with scikit-learn's `SimpleImputer`.

 **Dataset used:** `student_performance_raw.csv`


In [1]:
# Imports and reproducibility
import numpy as np
import os
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
# Load the supplied dataset
# Resolve the dataset path for both local execution and the course repository.
candidate_paths = [
    "data/raw/student_performance_raw.csv",         
    "../data/raw/student_performance_raw.csv",     
    "student_performance_raw.csv",                  
]

DATA_PATH = next((path for path in candidate_paths if os.path.exists(path)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "student_performance_raw.csv was not found. Expected it in data/raw/, "
        "../data/raw/, or beside this notebook."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from: {DATA_PATH}")

print(f"Dataset shape: {df.shape}")
display(df.head())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count").query("missing_count > 0"))


Loaded dataset from: student_performance_raw.csv
Dataset shape: (600, 17)


,student_id,city,city_tier,course,batch_type,age,enrollment_date,attendance_pct,weekly_study_hours,income_bracket,prev_exam_score,mock_test_1,mock_test_2,mock_test_3,doubt_sessions_attended,feedback_text,final_score
0,1447,Patna,2,JEE Main,Weekday,18,2026-05-27,91.1000,0.8000,<5L,65.2000,68.1000,59.0000,72.2000,1,Need more practice sheets for weak topics,42.0000
1,1405,Patna,2,NEET,Weekday,16,2024-07-08,91.9000,2.4000,NaN,64.9000,82.0000,88.4000,100.0000,3,Would like more one-on-one mentoring,62.0000
2,1510,Mumbai,1,JEE Main,Weekday,18,2025-03-17,67.1000,6.7000,5-10L,90.1000,86.5000,91.1000,76.4000,5,"Great teaching pace, doubts cleared quickly",56.4000
3,1456,Delhi,1,NEET,Weekday,18,2025-06-16,70.7000,2.0000,10-20L,29.7000,19.4000,24.9000,NaN,3,Need more practice sheets for weak topics,31.7000
4,1202,Patna,2,JEE Advanced,Weekday,17,2026-04-16,69.3000,6.5000,>20L,70.0000,63.8000,74.5000,69.1000,4,Need more practice sheets for weak topics,44.5000



Data types:


,dtype
student_id,int64
city,object
city_tier,int64
course,object
batch_type,object
age,int64
enrollment_date,object
attendance_pct,float64
weekly_study_hours,float64
income_bracket,object



Missing values:


,missing_count
weekly_study_hours,36
income_bracket,30
prev_exam_score,25
mock_test_3,21
feedback_text,68


## 1. Dataset Missingness and Column Types


In [3]:
# Confirm the supplied dataset's missingness profile
missing_summary = (
    df.isna().sum()
      .rename("missing_count")
      .to_frame()
      .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df))
      .query("missing_count > 0")
)

display(missing_summary)


,missing_count,missing_pct
weekly_study_hours,36,6.0000
income_bracket,30,5.0000
prev_exam_score,25,4.1667
mock_test_3,21,3.5000
feedback_text,68,11.3333


In [ ]:
class CustomImputer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        numeric_strategy="median",
        categorical_strategy="most_frequent",
        add_missing_indicator=True,
        column_overrides=None,
    ):

        if numeric_strategy not in {"mean", "median"}:
            raise ValueError("numeric_strategy must be 'mean' or 'median'.")
        if categorical_strategy != "most_frequent":
            raise ValueError("categorical_strategy must be 'most_frequent'.")
        if column_overrides is not None and not isinstance(column_overrides, dict):
            raise TypeError("column_overrides must be a dictionary or None.")

        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _get_fill_value(self, series, strategy):

        if strategy == "mean":
            return series.mean()
        if strategy == "median":
            return series.median()
        if strategy == "most_frequent":
            mode = series.mode(dropna=True)
            if mode.empty:
                return np.nan
            return mode.iloc[0]
        raise ValueError(f"Unsupported strategy '{strategy}'.")

    def fit(self, X, y=None):

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        self.feature_names_in_ = X.columns.tolist()
        self.column_types_ = {}
        self.fill_values_ = {}
        self.missing_indicator_columns_ = []

        for column in X.columns:
            series = X[column]

            if pd.api.types.is_numeric_dtype(series):
                self.column_types_[column] = "numeric"
                default_strategy = self.numeric_strategy
            else:
                self.column_types_[column] = "categorical"
                default_strategy = self.categorical_strategy

            strategy = (
                self.column_overrides.get(column, default_strategy)
                if self.column_overrides is not None
                else default_strategy
            )

            if strategy not in {"mean", "median", "most_frequent"}:
                raise ValueError(
                    f"Unsupported strategy '{strategy}' for column '{column}'."
                )

            if strategy in {"mean", "median"} and not pd.api.types.is_numeric_dtype(series):
                raise ValueError(
                    f"Strategy '{strategy}' requires a numeric column; "
                    f"'{column}' is non-numeric."
                )

            self.fill_values_[column] = self._get_fill_value(series, strategy)

            if series.isna().any():
                self.missing_indicator_columns_.append(column)

        return self

    def transform(self, X):

        check_is_fitted(
            self,
            attributes=["fill_values_", "feature_names_in_", "column_types_"]
        )

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        missing_columns = [c for c in self.feature_names_in_ if c not in X.columns]
        if missing_columns:
            raise ValueError(
                "Transform data is missing columns seen during fit: "
                + ", ".join(missing_columns)
            )

        extra_columns = [c for c in X.columns if c not in self.feature_names_in_]
        if extra_columns:
            raise ValueError(
                "Transform data contains columns not seen during fit: "
                + ", ".join(extra_columns)
                + ". Refit the imputer or align the schema before transforming."
            )

        X_out = X.copy()

        # Indicators must capture missingness BEFORE values are filled.
        if self.add_missing_indicator:
            for column in self.missing_indicator_columns_:
                X_out[f"{column}_was_missing"] = X_out[column].isna().astype(int)

        for column, fill_value in self.fill_values_.items():
            if pd.isna(fill_value):
                # An entirely missing training column cannot be meaningfully
                # imputed with mean/median/mode. Keep NaNs so the issue is explicit.
                continue
            X_out[column] = X_out[column].fillna(fill_value)

        return X_out


In [5]:
# Guard-rail test: transform before fit must fail clearly
unfitted_imputer = CustomImputer()

try:
    unfitted_imputer.transform(df)
except Exception as exc:
    print(f"Guard-rail passed: {type(exc).__name__}: {exc}")


Guard-rail passed: NotFittedError: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 2. Train/Test Split — Strict Train-Only Fitting

The custom imputer is fitted **only on the training split**. Test data is transformed afterward using the statistics learned from training data. This prevents test-set information from influencing the imputation values.


In [6]:
# 80/20 split
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Full dataset: {df.shape}")
print(f"Training split: {train_df.shape}")
print(f"Test split: {test_df.shape}")


Full dataset: (600, 17)
Training split: (480, 17)
Test split: (120, 17)


In [7]:
# Fit only on the training split
imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
)

imputer.fit(train_df)

print("Fitted columns:")
print(imputer.feature_names_in_)
print("\nAutomatically detected column types:")
display(pd.Series(imputer.column_types_, name="detected_type").to_frame())
print("\nLearned fill values:")
display(pd.Series(imputer.fill_values_, name="fill_value").to_frame())
print("\nTraining columns with missing values (indicator columns):")
print(imputer.missing_indicator_columns_)


Fitted columns:
['student_id', 'city', 'city_tier', 'course', 'batch_type', 'age', 'enrollment_date', 'attendance_pct', 'weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_1', 'mock_test_2', 'mock_test_3', 'doubt_sessions_attended', 'feedback_text', 'final_score']

Automatically detected column types:


,detected_type
student_id,numeric
city,categorical
city_tier,numeric
course,categorical
batch_type,categorical
age,numeric
enrollment_date,categorical
attendance_pct,numeric
weekly_study_hours,numeric
income_bracket,categorical



Learned fill values:


,fill_value
student_id,1282.0000
city,Mumbai
city_tier,1.0000
course,NEET
batch_type,Weekday
age,17.0000
enrollment_date,2024-10-23
attendance_pct,78.3500
weekly_study_hours,5.0000
income_bracket,5-10L



Training columns with missing values (indicator columns):
['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']


In [8]:
# Transform both splits using the SAME fitted imputer
train_imputed = imputer.transform(train_df)
test_imputed = imputer.transform(test_df)

print("Transformed train shape:", train_imputed.shape)
print("Transformed test shape:", test_imputed.shape)

indicator_cols = [f"{c}_was_missing" for c in imputer.missing_indicator_columns_]
print("\nIndicator columns created:")
print(indicator_cols)


Transformed train shape: (480, 22)
Transformed test shape: (120, 22)

Indicator columns created:
['weekly_study_hours_was_missing', 'income_bracket_was_missing', 'prev_exam_score_was_missing', 'mock_test_3_was_missing', 'feedback_text_was_missing']


## 3. Verification — No Missing Values Remain


In [9]:
# Programmatic verification
original_columns = imputer.feature_names_in_

train_missing_after = train_imputed[original_columns].isna().sum()
test_missing_after = test_imputed[original_columns].isna().sum()

print("Remaining missing values in original columns — TRAIN:")
display(train_missing_after[train_missing_after > 0].to_frame("missing_count"))

print("Remaining missing values in original columns — TEST:")
display(test_missing_after[test_missing_after > 0].to_frame("missing_count"))

assert train_imputed[original_columns].isna().sum().sum() == 0
assert test_imputed[original_columns].isna().sum().sum() == 0

print("PASS: No missing values remain in any original column after imputation.")


Remaining missing values in original columns — TRAIN:


,missing_count


Remaining missing values in original columns — TEST:


,missing_count


PASS: No missing values remain in any original column after imputation.


## 4. Missingness Indicators

For each column that had missing values **in the training data**, the transformer creates `<column>_was_missing`.

A value of `1` means that the original observation was missing; `0` means it was observed. This preserves information about missingness that would otherwise disappear after filling the missing value.


In [10]:
# Inspect indicator behavior
indicator_summary = pd.DataFrame({
    "indicator_column": indicator_cols,
    "train_missing_count": [train_df[c].isna().sum() for c in imputer.missing_indicator_columns_],
    "indicator_sum_after_transform": [train_imputed[c].sum() for c in indicator_cols],
})

display(indicator_summary)

for source_col, indicator_col in zip(imputer.missing_indicator_columns_, indicator_cols):
    assert train_imputed[indicator_col].sum() == train_df[source_col].isna().sum()
    assert set(train_imputed[indicator_col].unique()).issubset({0, 1})

print("PASS: Missing indicators correctly preserve the original missingness pattern.")


,indicator_column,train_missing_count,indicator_sum_after_transform
0,weekly_study_hours_was_missing,25,25
1,income_bracket_was_missing,24,24
2,prev_exam_score_was_missing,23,23
3,mock_test_3_was_missing,15,15
4,feedback_text_was_missing,54,54


PASS: Missing indicators correctly preserve the original missingness pattern.


## 5. Numeric Descriptive Statistics — Before vs After Imputation



In [11]:
numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()

stats_rows = []
for col in numeric_cols:
    before_mean = train_df[col].mean()
    before_std = train_df[col].std()
    after_mean = train_imputed[col].mean()
    after_std = train_imputed[col].std()

    stats_rows.append({
        "column": col,
        "before_mean": before_mean,
        "after_mean": after_mean,
        "mean_change": after_mean - before_mean,
        "before_std": before_std,
        "after_std": after_std,
        "std_change": after_std - before_std,
        "missing_in_train": train_df[col].isna().sum(),
    })

stats_comparison = pd.DataFrame(stats_rows)
display(stats_comparison)

print("Comment:")
print("Columns with missing values change because their missing observations are replaced "
      "with the training-set median. Columns with no missing values remain unchanged.")


,column,before_mean,after_mean,mean_change,before_std,after_std,std_change,missing_in_train
0,student_id,1290.5625,1290.5625,0.0000,173.8254,173.8254,0.0000,0
1,city_tier,1.4021,1.4021,0.0000,0.4908,0.4908,0.0000,0
2,age,16.9563,16.9563,0.0000,1.7315,1.7315,0.0000,0
3,attendance_pct,78.5463,78.5463,0.0000,13.4596,13.4596,0.0000,0
4,weekly_study_hours,6.0642,6.0088,-0.0554,4.3503,4.2418,-0.1084,25
5,prev_exam_score,65.8245,65.8377,0.0132,14.3708,14.0217,-0.3491,23
6,mock_test_1,65.9452,65.9452,0.0000,16.5662,16.5662,0.0000,0
7,mock_test_2,67.8863,67.8863,0.0000,18.1088,18.1088,0.0000,0
8,mock_test_3,70.7015,70.7202,0.0187,19.0773,18.7765,-0.3008,15
9,doubt_sessions_attended,4.0042,4.0042,0.0000,2.0259,2.0259,0.0000,0


Comment:
Columns with missing values change because their missing observations are replaced with the training-set median. Columns with no missing values remain unchanged.


## 6. Sanity Check Against scikit-learn `SimpleImputer`


In [12]:
# Equivalent SimpleImputer on numeric columns, fitted on TRAIN only
numeric_train = train_df[numeric_cols]

sk_imputer = SimpleImputer(strategy="median")
sk_imputer.fit(numeric_train)

custom_numeric_values = np.array([imputer.fill_values_[c] for c in numeric_cols], dtype=float)
sk_numeric_values = sk_imputer.statistics_.astype(float)

comparison = pd.DataFrame({
    "column": numeric_cols,
    "custom_fill_value": custom_numeric_values,
    "simple_imputer_value": sk_numeric_values,
    "absolute_difference": np.abs(custom_numeric_values - sk_numeric_values),
})

display(comparison)

assert np.allclose(custom_numeric_values, sk_numeric_values, equal_nan=True)

print("PASS: CustomImputer and SimpleImputer produce identical numeric median fill values.")


,column,custom_fill_value,simple_imputer_value,absolute_difference
0,student_id,1282.0000,1282.0000,0.0000
1,city_tier,1.0000,1.0000,0.0000
2,age,17.0000,17.0000,0.0000
3,attendance_pct,78.3500,78.3500,0.0000
4,weekly_study_hours,5.0000,5.0000,0.0000
5,prev_exam_score,66.1000,66.1000,0.0000
6,mock_test_1,66.5500,66.5500,0.0000
7,mock_test_2,66.3500,66.3500,0.0000
8,mock_test_3,71.3000,71.3000,0.0000
9,doubt_sessions_attended,4.0000,4.0000,0.0000


PASS: CustomImputer and SimpleImputer produce identical numeric median fill values.


## 7. Categorical Imputation Check

The default categorical strategy is `most_frequent`. This section verifies that the categorical fill values learned by the custom class correspond to the training-set mode.

`feedback_text` is treated as a non-numeric/object column by the automatic type detector, so it follows the same categorical rule.


In [13]:
categorical_cols = train_df.select_dtypes(exclude=np.number).columns.tolist()

categorical_check = []
for col in categorical_cols:
    mode_value = train_df[col].mode(dropna=True).iloc[0]
    categorical_check.append({
        "column": col,
        "custom_fill_value": imputer.fill_values_[col],
        "training_mode": mode_value,
        "matches": imputer.fill_values_[col] == mode_value,
    })

categorical_check_df = pd.DataFrame(categorical_check)
display(categorical_check_df)

assert categorical_check_df["matches"].all()
print("PASS: All categorical fill values match the training-set mode.")


,column,custom_fill_value,training_mode,matches
0,city,Mumbai,Mumbai,True
1,course,NEET,NEET,True
2,batch_type,Weekday,Weekday,True
3,enrollment_date,2024-10-23,2024-10-23,True
4,income_bracket,5-10L,5-10L,True
5,feedback_text,Need more practice sheets for weak topics,Need more practice sheets for weak topics,True


PASS: All categorical fill values match the training-set mode.


# 8. Reflection Questions

### 1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?

`fit()` learns the values used to replace missing observations, so fitting on the full dataset would allow information from the test set to influence those statistics. That is a form of data leakage because the preprocessing step would have access to information from data that is supposed to simulate unseen future data. Fitting only on the training split keeps the test set completely unseen during model preparation. The already-fitted imputer can then transform both training and test data consistently without recalculating any statistics.

### 2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem for this column? What does the `add_missing_indicator` feature contribute that plain imputation does not?

No, mean or median imputation does not genuinely solve the underlying MNAR problem because the probability of missingness is related to the unobserved value itself. Replacing missing `mock_test_3` values with a typical value can reduce the visible effect of the missingness mechanism and distort the distribution. The missingness indicator preserves an additional signal showing which observations were originally missing. A downstream model can therefore learn that being missing may itself carry predictive information, although the indicator does not by itself remove the bias caused by MNAR missingness.

### 3. Suppose a brand-new column, entirely missing in the training data but present in the test data, is passed to your imputer. What does your current implementation do — and what SHOULD a production-grade version do instead?

In the current implementation, a truly new column that was not seen during fit() causes transform() to raise a clear ValueError because the input schema does not match the training schema. This is intentional because the imputer has no learned fill value or strategy for an unseen feature. If a column existed during training but was entirely missing, its learned statistic would be NaN because a meaningful mean, median, or mode could not be calculated. A production-grade version should enforce an explicit schema policy, such as rejecting unexpected columns, aligning them to a predefined schema, dropping unusable features, or applying a documented domain-specific fallback value.


# 9. Optional Bonus — Per-Column Strategy Overrides

```python
CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    column_overrides={
        "weekly_study_hours": "mean",
        "income_bracket": "most_frequent",
    }
)
```

The next cell demonstrates two different per-column strategies: `weekly_study_hours` uses **mean**, while `prev_exam_score` uses **median**.


In [14]:
# Bonus demonstration: two columns with different strategies
bonus_imputer = CustomImputer(
    numeric_strategy="median",
    categorical_strategy="most_frequent",
    add_missing_indicator=True,
    column_overrides={
        "weekly_study_hours": "mean",
        "prev_exam_score": "median",
    },
)

bonus_imputer.fit(train_df)
bonus_train = bonus_imputer.transform(train_df)

bonus_demo = pd.DataFrame({
    "column": ["weekly_study_hours", "prev_exam_score"],
    "strategy_used": ["mean", "median"],
    "learned_fill_value": [
        bonus_imputer.fill_values_["weekly_study_hours"],
        bonus_imputer.fill_values_["prev_exam_score"],
    ],
    "expected_value": [
        train_df["weekly_study_hours"].mean(),
        train_df["prev_exam_score"].median(),
    ],
})

display(bonus_demo)

assert np.isclose(
    bonus_imputer.fill_values_["weekly_study_hours"],
    train_df["weekly_study_hours"].mean()
)
assert np.isclose(
    bonus_imputer.fill_values_["prev_exam_score"],
    train_df["prev_exam_score"].median()
)

assert bonus_train[original_columns].isna().sum().sum() == 0

print("PASS: Per-column strategy overrides work with two different strategies.")


,column,strategy_used,learned_fill_value,expected_value
0,weekly_study_hours,mean,6.0642,6.0642
1,prev_exam_score,median,66.1000,66.1000


PASS: Per-column strategy overrides work with two different strategies.
